# Plana_Riviere - consolidation des chroniques

Une seule sonde : **CTD** (Diver autonome : niveau, conductivité, température).
Pas de choix de sonde, pas de fusion, pas de points de contrôle.

Les fonctions communes sont dans la librairie `ouysse`. Ce notebook ne garde que
ce qui est propre à la station : chemins, mesures écartées, réglages du filtre.

## 1. Imports

In [ ]:
import os
import numpy as np
import unicodedata
import pandas as pd
import matplotlib.pyplot as plt
import ouysse
from ouysse import *
print("ouysse-hydro", ouysse.__version__)

## 2. Chemins d'accès

In [ ]:
BASE        = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Rivière Planagrèze\Gaetan"
CTD_PATH    = os.path.join(BASE, r"Données brutes")
BARO_PATH   = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\1 - Données BARO\Gourdon baro\Patm Calès et Thémines.xlsx"
PLUIE_PATH  = r"Y:\MISSIONS\Eau\1 - Projet Hydrogéologique Ouysse\3 - Hydrodynamique\0 - Stations en continu\Pluie_BV_Ouysse.csv"

UTC_CTD_PATH     = os.path.join(BASE, "UTC_CTD.xlsx")
OLDDATA_PATH     = os.path.join(BASE, r"données consolidées\PlanaRiv_Old.xlsx")
SORTIE_CONSOLIDE = os.path.join(BASE, r"données consolidées\Plana_Riviere_consolide.xlsx")
SORTIE_FINALE    = os.path.join(BASE, r"données consolidées\Plana_Riviere_final.xlsx")
SORTIE_SVG       = os.path.join(BASE, r"données consolidées\Graphes.svg")

PREFIXE_CTD = "Plana_Riviere"
COL_UTC     = "UTC Fichier"     
BARO_COL    = "Patm Thémines [hPa]"
PAS         = "1h"

PARAMETRES = ["Niveau_(cm)", "Conductivité", "Température"]

## 3. CTD : lecture, UTC et compensation barométrique

In [ ]:
metadata = pd.read_excel(UTC_CTD_PATH)
baro = pd.read_excel(BARO_PATH)[["DATE", BARO_COL]]
baro["DATE"] = pd.to_datetime(baro["DATE"], errors="coerce")
baro = baro.dropna(subset=["DATE"]).drop_duplicates("DATE")

morceaux = []
for nom in fichiers(CTD_PATH, PREFIXE_CTD):
    try:
        CTD, decalage = en_utc(lire_CTD(nom, CTD_PATH, PAS), nom,
                               metadata, col_utc=COL_UTC)
    except Exception as e:
        print(f"  IGNORÉ  {nom} : {e}")
        continue
    m = pd.merge(CTD, baro, left_on="Date/time", right_on="DATE", how="left")
    m["Niveau_(cm)"] = m["Pression[cmH2O]"] - m[BARO_COL] * HPA_EN_CMH2O
    m = m.rename(columns={"Cond_(µS/cm)": "Conductivité", "Température[°C]": "Température"})
    morceaux.append(m[["Date/time"] + PARAMETRES])
    print(f"  {nom:45s} UTC+{decalage:g} vers UTC   ({len(CTD)} lignes)")

merge_ctd_df = pd.concat(morceaux, ignore_index=True).sort_values("Date/time", kind="stable")
merge_ctd_df["DATE"] = merge_ctd_df["Date/time"]
print(f"\n{len(morceaux)} campagne(s), {len(merge_ctd_df)} enregistrements.")

## 4. Raccordement à l'ancienne chronique

L'ancien fichier consolidé et les campagnes récentes sont la **même sonde CTD**,
séparées par un trou d'exploitation : le décalage est mesuré à la jonction et
appliqué aux campagnes, pour que la chronique soit continue.

In [ ]:
RENOMMAGE_OLD = {
    "Date/time": "DATE",          # <- ta colonne réelle, absente de l'ancienne version
    "NIVEAU": "Niveau_(cm)",
    "CONDUCTIVITE": "Conductivité",
    "TEMPERATURE CTD": "Température",
    "Cond_(µS/cm)": "Conductivité",
    "Temp_(°C)": "Température",
}

norme = lambda c: unicodedata.normalize("NFKC", str(c)).strip()
olddata_df = pd.read_excel(OLDDATA_PATH)
olddata_df.columns = [norme(c) for c in olddata_df.columns]
olddata_df = olddata_df.rename(columns={norme(k): v for k, v in RENOMMAGE_OLD.items()})

merge_ctd_df = raccorder_campagnes(olddata_df, merge_ctd_df, [
    ("Niveau", "Niveau_(cm)", "Niveau_(cm)", "cm"),
    ("Conductivité", "Conductivité", "Conductivité", "µS/cm")])


## 5. Assemblage sur la grille horaire

In [ ]:
full_data = sur_grille([empiler([olddata_df, merge_ctd_df], PARAMETRES)], PAS)

print("Hors gamme physique :")
full_data = appliquer_gammes(full_data)

full_data.to_excel(SORTIE_CONSOLIDE)
print(f"\nfichier fusionné : {SORTIE_CONSOLIDE}")

## 6. Corrections capteur

`VOIES_ECARTEES` met des mesures à l'écart. Il n'y a pas de sonde de secours ici :
la lacune reste, et l'interpolation ne comblera pas plus de 12 h.

In [ ]:
#: (début, fin, colonne, motif) : mesures mises à l'écart.
VOIES_ECARTEES = [
]

print("Voies écartées :")
full_data = ecarter(full_data, VOIES_ECARTEES)

## 7. Filtre IQR et lissage

In [ ]:
FENETRE_IQR, K_IQR = "24h", 0   # k = 0 : pas de filtre
LISSAGE_H = 6                     # 0 = pas de lissage ; sinon médiane glissante, en heures

avant = full_data["Conductivité"]
full_data["Conductivité"] = filtre_iqr(avant, FENETRE_IQR, K_IQR, lissage_h=LISSAGE_H)
full_data["Conductivité_Moyenne_Mobile"] = full_data["Conductivité"].rolling("6h", center=True).mean()

graphe([(avant, "avant IQR et lissage", "darkorange"),
        (full_data["Conductivité"], "après IQR et lissage", "black")],
       titre="Conductivité", ylab="Conductivité (µS/cm)")

## 8. Cote NGF, interpolation et statuts

Les lacunes de moins de 12 h sont comblées. `Statut_<grandeur>` dit si la valeur
est mesurée, interpolée ou manquante.

In [ ]:
NIVEAU_NGF = None       # cote du zéro de l'échelle, None si elle n'est pas connue
MAX_TROU_H = 12

full_data = interpoler_avec_statut(full_data, PARAMETRES, MAX_TROU_H, PAS)

if NIVEAU_NGF is not None:
    full_data["Niveau_(mNGF)"] = NIVEAU_NGF + full_data["Niveau_(cm)"] / 100
    full_data["Statut_Niveau_(mNGF)"] = full_data["Statut_Niveau_(cm)"]
    print(f"Zéro de l'échelle à {NIVEAU_NGF:.4f} m NGF")
display(pd.DataFrame({c: full_data[f"Statut_{c}"].value_counts()
                      for c in PARAMETRES}).fillna(0).astype(int).T)

## 9. Sauvegarde et graphe de synthèse

In [ ]:
finaux = [c for c in PARAMETRES + ["Niveau_(mNGF)"] if c in full_data]
colonnes = [c for p in finaux for c in (p, f"Statut_{p}") if c in full_data]
sortie = full_data[colonnes].copy()
sortie.attrs["ouysse"] = ouysse.__version__
sortie.to_excel(SORTIE_FINALE)
print(f"{SORTIE_FINALE} : {len(sortie)} pas x {len(colonnes)} colonnes "
      f"(ouysse-hydro {ouysse.__version__})")

graphe_synthese(full_data, "Niveau_(cm)", "Niveau (cm)", pluie=PLUIE_PATH, sortie=SORTIE_SVG, marge_jours=15)

In [ ]:
graphe_statuts(full_data, finaux)